## **CPU RAM and GPU VRAM**

In `CUDA` programming, we often deal with two types of memory: `CPU RAM` (Random Access Memory) and `GPU VRAM` (Video Random Access Memory).

CPU RAM is called `host memory`, while GPU VRAM is called `device memory`. 

Whenever we want to perform computations on the GPU, we need to transfer data from the CPU RAM to the GPU VRAM. This process is known as `memory transfer` and can be a bottleneck in performance if not managed properly.

> Note:

- To avoid inconsistencies which variable is stored where when writing CUDA code, we use `h_` prefix for variables stored in host memory (CPU RAM) 

- We use `d_` prefix for variables stored in device memory (GPU VRAM)

<hr>


**why can't we just write directly to VRAM?**

### The Background: The City Map of Your Computer
Imagine your computer as a city. 
* **The CPU (Ryzen 5)** is the Mayor/Central Government. It runs the Operating System (Linux) and controls everything.
* **The CPU RAM (System Memory)** is the government's local archive. It is located physically inches away from the CPU, connected by an ultra-fast, ultra-low-latency bus.
* **The GPU (RTX 3060)** is a massive, specialized factory located on the outskirts of the city.
* **The GPU VRAM (Device Memory)** is the factory's private warehouse. It is extremely fast for the factory workers (GPU cores) to access, but it is physically separate from the CPU.
* **The PCIe Bus (Peripheral Component Interconnect Express)** is the highway connecting the CPU/RAM to the GPU/VRAM. 

### So, Why Do We Write to RAM First?

#### Reason 1: The Origin of Data
Where does data come from? 
In our vector addition code, the data was generated by the CPU using a `for` loop and `rand()`. Because the CPU is executing those instructions, it has to store the output somewhere. The fastest place for the CPU to write is its own RAM. 

If you are loading a 3D model, an image, or a dataset for AI, that data lives on your Hard Drive/SSD. The Operating System (which runs on the CPU) reads the disk and puts the file into CPU RAM. The GPU cannot natively talk to your hard drive (traditionally). **All data entering the system must go through the CPU and its RAM first.**

#### Reason 2: The PCIe Highway (Latency vs. Bandwidth)
Technically, your CPU *can* see a portion of the GPU's VRAM over the PCIe highway. So why not just have the CPU write the `rand()` numbers directly into VRAM, skipping the CPU RAM entirely?

Because of **Latency** (the time it takes for a single piece of data to travel).
* Writing an integer from the CPU to **CPU RAM** takes about **50 nanoseconds**.
* Writing an integer from the CPU to **GPU VRAM** over the PCIe bus takes **thousands of nanoseconds** because the signal has to travel across the motherboard.

If you made the CPU write 65,536 integers directly into VRAM one by one inside a `for` loop, the CPU would have to wait for each integer to travel down the highway. Your program would grind to a halt.

**The Solution: Bulk Shipping (`cudaMemcpy`)**
The PCIe highway is terrible at sending 65,536 individual cars (high latency). But it is *incredible* at sending one massive semi-truck carrying 65,536 items (high bandwidth). 
1. The CPU builds the entire array quickly in its local RAM.
2. We call `cudaMemcpy`.
3. A dedicated piece of hardware (the DMA controller) picks up the entire array from RAM and blasts it across the PCIe highway to the VRAM in one giant, ultra-fast chunk. 

---

### Wait... Can we write to VRAM directly? (What you are missing)

You actually *can*, and NVIDIA knows that writing `cudaMalloc` and `cudaMemcpy` for every single array gets exhausting. 

As CUDA evolved, NVIDIA introduced advanced memory concepts to solve exactly what you are asking about:

#### 1. Unified Memory (The modern way to code)
Introduced in CUDA 6, NVIDIA created **Unified Memory**. Instead of writing separate `h_a` and `d_a`, you can allocate memory using `cudaMallocManaged()`. 
* It creates a single pointer (e.g., `int *a`).
* Both the CPU and the GPU can read and write to this exact same pointer!
* **How it works:** Under the hood, the CUDA driver acts like a magician. If the CPU tries to read it, the driver secretly moves the data over the PCIe bus to the RAM. If the GPU tries to read it, the driver secretly pages it over to the VRAM. 
* *You don't have to write `cudaMemcpy` at all.* 

*(CoffeeBeforeArch will actually teach you Unified Memory a few videos into the playlist!)*

#### 2. Zero-Copy Memory / Mapped Memory
You can actually instruct the GPU to *not* copy data to VRAM, but instead reach across the PCIe bus and read the data directly out of the CPU's RAM while it does its math. This is called Zero-Copy memory. It's useful if the data is so massive it won't fit in your 6GB RTX 3060 VRAM, or if you only need to read the data exactly one time.

#### 3. GPUDirect Storage / Microsoft DirectStorage (Cutting Edge)
Remember how I said the GPU can't talk to the SSD? That is finally changing. In the latest AI server farms and modern AAA games, a new technology allows the NVMe SSD to stream data *directly* to the GPU VRAM over the PCIe bus, completely bypassing the CPU and CPU RAM. 

### Summary
You **write to RAM first and copy to VRAM** because the CPU is the brain that generates or loads the data, and the PCIe bus connecting them requires data to be sent in massive, pre-packaged chunks for good performance. 